# 04. LifeMate AI Agent — Huấn Luyện QLoRA & Tạo Bài Nộp trên Kaggle (GPU P100)
**Zalo AI Challenge 2026** — SFT Trajectory Fine-Tuning Pipeline

### ⚙️ CÀI ĐẶT BẮT BUỘC TRÊN KAGGLE (Thanh bên phải -> Notebook options):
1. **ACCELERATOR:** Chọn **GPU P100** (Khuyên dùng - 16GB VRAM, băng thông cao, không bị nghẽn bộ nhớ) hoặc **GPU T4 x 2**.
2. **INTERNET:** Bắt buộc bật **INTERNET = ON** (để clone repo GitHub và nạp base model Hugging Face).
3. **DATA INPUT:** Bấm nút **+ Add Input** ở góc trên bên phải để nạp dataset `train+public_test` (nếu đã tạo dataset trên Kaggle).

---
- **Mô hình nền:** `Qwen/Qwen3-4B-Instruct-2507` (4.05B params < 5.0B)
- **Kỹ thuật:** 4-bit QLoRA (NF4) + SDPA Attention + Prompt Loss Masking (`labels = -100`)
- **Thời gian huấn luyện:** ~12 - 18 phút trên GPU P100 (tốc độ ~0.5s/bước)
- **Mục tiêu:** Đạt độ chuẩn xác gọi tool tối đa, triệt tiêu lặp lệnh, đạt điểm 75 - 85+!

## 1. Kiểm tra Card Đồ Họa (GPU P100 Status)

In [ ]:
!nvidia-smi

## 2. Cài đặt các thư viện AI cần thiết

In [ ]:
!pip install -q "transformers>=4.45" accelerate peft bitsandbytes datasets sentencepiece huggingface_hub

## 3. Clone Mã Nguồn vào `/kaggle/working/LifeMateAgent`

In [ ]:
import os
from pathlib import Path

WORKSPACE = Path("/kaggle/working/LifeMateAgent")
if not WORKSPACE.exists():
    !git clone https://github.com/Platypus27-coder/zalo-ai-challenge.git {WORKSPACE}

%cd {WORKSPACE}
!git checkout main
!git pull origin main

## 4. Chuẩn bị Dữ Liệu Train & Test
Hệ thống sẽ tự động quét thư mục `/kaggle/input/` để phát hiện dữ liệu (kể cả đã giải nén hay file `.zip`).

In [ ]:
import os, shutil
from pathlib import Path

DATA_DIR = WORKSPACE / "train+public_test"
DATA_DIR.mkdir(parents=True, exist_ok=True)

# 1. Tìm và liên kết thư mục train/ và public_test/ nếu Kaggle đã giải nén sẵn
if Path("/kaggle/input").exists():
    found_train = list(Path("/kaggle/input").rglob("train.jsonl"))
    found_test = list(Path("/kaggle/input").rglob("public_test.jsonl"))

    if found_train and not (DATA_DIR / "train" / "train.jsonl").exists():
        train_src = found_train[0].parent
        target_train = DATA_DIR / "train"
        try:
            os.symlink(str(train_src), str(target_train))
            print(f"🔗 Đã liên kết thư mục train từ: {train_src}")
        except Exception:
            !cp -r "{train_src}" "{target_train}"

    if found_test and not (DATA_DIR / "public_test" / "public_test.jsonl").exists():
        test_src = found_test[0].parent
        target_test = DATA_DIR / "public_test"
        try:
            os.symlink(str(test_src), str(target_test))
            print(f"🔗 Đã liên kết thư mục public_test từ: {test_src}")
        except Exception:
            !cp -r "{test_src}" "{target_test}"

# 2. Nếu dữ liệu dạng file zip trong /kaggle/input, tự động giải nén
if not (DATA_DIR / "train" / "train.jsonl").exists() and Path("/kaggle/input").exists():
    zips = list(Path("/kaggle/input").rglob("*.zip"))
    for z in zips:
        if "train" in z.name.lower() or "lifemate" in z.name.lower() or "test" in z.name.lower():
            print(f"📦 Đang giải nén zip từ: {z}")
            !unzip -q -o "{z}" -d {WORKSPACE}/
            break

# Kiểm tra tính sẵn sàng
train_ok = (DATA_DIR / "train" / "train.jsonl").exists()
test_ok = (DATA_DIR / "public_test" / "public_test.jsonl").exists()

if train_ok and test_ok:
    print("\n✅ Dữ liệu train (1.545 kịch bản) và public_test (634 mẫu) đã sẵn sàng 100%!")
elif train_ok:
    print("\n✅ Dữ liệu train (1.545 kịch bản) đã sẵn sàng!")
else:
    print("\n⚠️ Chưa thấy train.jsonl trong /kaggle/input!")
    print("👉 Cách nạp: Bấm '+ Add Input' ở góc trên bên phải Kaggle để upload file train+public_test.zip")


## 5. Bóc tách 1.545 kịch bản thành 5.086 lượt huấn luyện SFT (ChatML)
Mỗi lượt huấn luyện mô phỏng Teacher-Forcing có mặt đầy đủ các thông tin ngữ cảnh, danh mục tool và phản hồi trợ lý.

In [ ]:
!python scripts/prepare_sft_data.py

## 6. Huấn luyện QLoRA 4-bit trên GPU P100 (Thời gian: ~12 - 18 phút)
- **Batch size = 1, Gradient Accumulation = 8** $\rightarrow$ Effective Batch Size = 8.
- Bộ nhớ VRAM chiếm chỉ **~4.5 GB** (GPU P100 thừa tận 11.5 GB VRAM).
- Không bị tràn bộ nhớ hay nghẽn swap sang CPU, tốc độ ổn định cực nhanh!

In [ ]:
ADAPTER_OUT = "/kaggle/working/qwen3_4b_lora"

!python scripts/train_lora.py \
    --epochs 2 \
    --lr 2e-4 \
    --batch-size 1 \
    --grad-accum 8 \
    --output-dir {ADAPTER_OUT}

## 7. Kiểm tra Adapter & Nén file zip cho Kaggle Output

In [ ]:
import shutil
from pathlib import Path

adapter_dir = Path(ADAPTER_OUT)
if (adapter_dir / "adapter_model.safetensors").exists():
    print(f"🎉 Huấn luyện thành công 100%! Trọng số lưu tại: {adapter_dir}")
    # Nén zip để tải về trực tiếp từ tab Output trên thanh bên phải Kaggle
    zip_path = Path("/kaggle/working/qwen3_4b_lora_adapter")
    shutil.make_archive(str(zip_path), "zip", str(adapter_dir))
    print(f"📦 File zip tải về: {zip_path}.zip")
else:
    print("❌ Chưa tìm thấy adapter_model.safetensors!")

## 8. Đẩy LoRA Adapter lên Hugging Face Hub (Khuyên dùng - Private Repo)
Đẩy adapter (~50MB) lên Hugging Face Hub để có thể tái sử dụng ở bất kỳ đâu chỉ với tên repo.

In [ ]:
# 8.1 Đăng nhập Hugging Face (Dán Write Token tại: https://huggingface.co/settings/tokens)
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
# 8.2 Đẩy Adapter lên Hugging Face (Repo Private)
HF_REPO_ID = "your_username/lifemate-qwen3-4b-lora"  # Thay bằng tên tài khoản Hugging Face của bạn

!python scripts/push_to_hub.py \
    --adapter-dir {ADAPTER_OUT} \
    --repo-id {HF_REPO_ID}

## 9. Chạy Tạo Bài Nộp Cho 634 Mẫu Public Test
Chạy suy luận toàn bộ 634 mẫu với Não bộ LoRA mới và bộ lọc Guardrails chống lặp/chống ảo giác.

In [ ]:
SUBMISSION_OUT = "/kaggle/working/submission_public_test.jsonl"

!python scripts/generate_submission.py \
    --split public_test \
    --asr-mode precomputed \
    --agent-mode transformers \
    --model-name Qwen/Qwen3-4B-Instruct-2507 \
    --adapter-path {ADAPTER_OUT} \
    --output {SUBMISSION_OUT} \
    --no-resume \
    --verbose

## 10. Kiểm tra Hợp Chuẩn 7 Lớp & Tải File Nộp Bài
Chạy kiểm tra linter chính thức để đảm bảo 100% hợp lệ trước khi gửi BTC.

In [ ]:
!python scripts/lint_submission.py {SUBMISSION_OUT}

print("\n" + "=" * 60)
print("🎉 CHÚC MỪNG BẠN! FILE NỘP BÀI ĐÃ SẴN SÀNG!")
print("👉 Hãy nhìn sang thanh bên phải (Kaggle sidebar) -> Mở tab 'Output'")
print("👉 Nhấp vào dấu 3 chấm cạnh file 'submission_public_test.jsonl' -> Chọn 'Download'")
print("👉 Nộp file vừa tải lên Leaderboard Zalo AI Challenge!")
print("=" * 60)